# 🟣 SCD Implementation – Players (Type 1 & Type 2)

## 📌 Objective

Handle changes in player data over time using Slowly Changing Dimensions.

## 🧠 Types Implemented

* SCD Type 1 → Overwrite
* SCD Type 2 → History tracking

## 🔁 Type 1

* Updates existing records
* No history maintained

## 🔁 Type 2

* Tracks historical changes
* Adds:

  * start_date
  * end_date
  * is_current flag

## ⚙️ Logic

* Detect changes in key attributes
* Expire old records
* Insert new version

## 📥 Source

* Bronze Players

## 📤 Target

* Silver Players (SCD Enabled)

## 🎯 Goal

Enable historical tracking for analytics


In [0]:
source_df = spark.read.table('game_analytics.bronze.players')
display(source_df)

### Testing SCD Type 1 by creating new record.

In [0]:
%sql
INSERT INTO game_analytics.bronze.players
VALUES (
    'P0151',
    'MyName',
    'Trump',
    '2026-04-12',
    '8',
    'USA',
    current_timestamp(),
    null,
    current_timestamp()
)

In [0]:
%sql
WITH latest_source AS (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY player_id
             ORDER BY ingestion_time DESC
           ) AS rn
    FROM game_analytics.bronze.players
  )
  WHERE rn = 1
)

MERGE INTO game_analytics.silver.players AS target
USING latest_source AS source
ON target.player_id = source.player_id

WHEN MATCHED THEN
UPDATE SET
  target.first_name = source.first_name,
  target.last_name  = source.last_name,
  target.level      = CAST(source.level AS INT),
  target.country    = source.country,
  target.created_at = source.created_at

WHEN NOT MATCHED THEN
INSERT (
  player_id,
  first_name,
  last_name,
  joined_date,
  level,
  country,
  created_at,
  ingestion_time
)
VALUES (
  source.player_id,
  source.first_name,
  source.last_name,
  source.joined_data,
  CAST(source.level AS INT),
  source.country,
  source.created_at,
  source.ingestion_time
)

In [0]:
%sql
SELECT *
FROM game_analytics.silver.players

In [0]:
%sql
ALTER TABLE game_analytics.silver.players
ADD COLUMNS (
  start_date DATE,
  end_date DATE,
  is_current BOOLEAN
);

In [0]:
%sql
UPDATE game_analytics.silver.players
SET 
  start_date = current_date(),
  end_date = NULL,
  is_current = true;

In [0]:
%sql
WITH latest_source AS (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY player_id
             ORDER BY ingestion_time DESC
           ) AS rn
    FROM game_analytics.bronze.players
  )
  WHERE rn = 1
)
MERGE INTO game_analytics.silver.players AS target
USING latest_source AS source
ON target.player_id = source.player_id AND target.is_current = true

-- 1️⃣ Expire old record
WHEN MATCHED AND (
    target.first_name != source.first_name OR
    target.last_name  != source.last_name OR
    target.level      != CAST(source.level AS INT) OR
    target.country    != source.country
)
THEN UPDATE SET
    target.end_date = current_date(),
    target.is_current = false

-- 2️⃣ Insert new record
WHEN NOT MATCHED
THEN INSERT (
  player_id,
  first_name,
  last_name,
  joined_date,
  level,
  country,
  created_at,
  ingestion_time,
  start_date,
  end_date,
  is_current
)
VALUES (
  source.player_id,
  source.first_name,
  source.last_name,
  source.joined_data,
  CAST(source.level AS INT),
  source.country,
  source.created_at,
  source.ingestion_time,
  current_date(),
  NULL,
  true
)

In [0]:
%sql
SELECT *
FROM game_analytics.silver.players